# 08 - Evaluacion final del modelo congelado

Este notebook evalua una unica configuracion final sobre el test temporal. No entrena modelos, no recalibra thresholds y no hace seleccion de variables. La finalidad es producir las metricas, figuras y tablas definitivas para la memoria del TFG.

## 1. Setup y rutas

Se cargan las rutas del proyecto desde `triaje_ia.config`. Los artefactos de entrada proceden del notebook 07 y los artefactos de texto/BERT del notebook 06. No se leen secretos ni se usan rutas absolutas.

In [ ]:
# ruff: noqa: E402, I001
import json
import sys
import warnings
from pathlib import Path
from typing import Any

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.metrics import (
    average_precision_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_recall_fscore_support,
)
from sklearn.preprocessing import label_binarize


def find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").exists() and (candidate / "src").exists():
            return candidate
    raise RuntimeError("No se pudo localizar la raiz del proyecto")


PROJECT_DISCOVERED = find_project_root(Path.cwd().resolve())
SRC_DIR = PROJECT_DISCOVERED / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from triaje_ia.config import (
    DATA_PROCESSED,
    FIGURES_DIR,
    MODELS_DIR,
    PROJECT_ROOT,
    REPORTS_DIR,
)

assert PROJECT_ROOT == PROJECT_DISCOVERED
warnings.filterwarnings("ignore")
RANDOM_STATE = 42
CLASSES = np.array([1, 2, 3, 4, 5])
CLASS_NAMES = [f"acuity_{i}" for i in CLASSES]

INPUTS = {
    "model": MODELS_DIR / "lgbm_bert_final.joblib",
    "thresholds": MODELS_DIR / "thresholds.json",
    "feature_list": MODELS_DIR / "feature_list.json",
    "metadata": MODELS_DIR / "model_training_metadata.json",
    "X_test": DATA_PROCESSED / "X_test.parquet",
    "y_test": DATA_PROCESSED / "y_test.parquet",
    "X_test_exp5b": DATA_PROCESSED / "X_test_exp5b.parquet",
    "llm_test": DATA_PROCESSED / "llm_features_test.parquet",
    "bert_test": DATA_PROCESSED / "bert_embeddings_test.parquet",
    "bert_svd": DATA_PROCESSED / "bert_svd.joblib",
    "bert_svd_legacy": DATA_PROCESSED / "svd_bert.joblib",
}

EVAL_DIR = REPORTS_DIR / "final_evaluation"
FIG_DIR = FIGURES_DIR / "final_evaluation"
OUTPUTS = {
    "confusion_matrix": FIG_DIR / "confusion_matrix.png",
    "confusion_matrix_normalized": FIG_DIR / "confusion_matrix_normalized.png",
    "shap_summary": FIG_DIR / "shap_summary.png",
    "classification_report": EVAL_DIR / "classification_report.csv",
    "final_metrics": EVAL_DIR / "final_metrics.json",
    "safety_analysis": EVAL_DIR / "safety_analysis.csv",
    "auprc_metrics": EVAL_DIR / "auprc_metrics.json",
}
EVAL_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

print(f"Proyecto: {PROJECT_ROOT}")
for name, path in OUTPUTS.items():
    print(f"{name:<28} -> {path.relative_to(PROJECT_ROOT)}")

## 2. Carga de artefactos finales

El modelo, la lista de variables y la politica de decision se cargan ya congelados. En este notebook se comprueba compatibilidad, pero no se cambia ningun parametro.

In [ ]:
required = [
    INPUTS["model"],
    INPUTS["thresholds"],
    INPUTS["feature_list"],
    INPUTS["metadata"],
    INPUTS["X_test"],
    INPUTS["y_test"],
]
missing = [path for path in required if not path.exists()]
if missing:
    raise FileNotFoundError(
        "Faltan artefactos finales: " + ", ".join(str(p) for p in missing)
    )

model = joblib.load(INPUTS["model"])
thresholds = json.loads(INPUTS["thresholds"].read_text(encoding="utf-8"))
feature_payload = json.loads(INPUTS["feature_list"].read_text(encoding="utf-8"))
training_metadata = json.loads(INPUTS["metadata"].read_text(encoding="utf-8"))

FINAL_FEATURES = feature_payload["features"]
THRESHOLD_A1 = float(thresholds["threshold_a1"])
CLASS_WEIGHTS = np.asarray(thresholds["class_weights"], dtype=float)

assert THRESHOLD_A1 >= 0.20
assert CLASS_WEIGHTS.shape == (5,)
assert len(FINAL_FEATURES) == int(feature_payload["n_features"])
assert not pd.Index(FINAL_FEATURES).duplicated().any()
assert model.n_features_in_ == len(FINAL_FEATURES)
assert thresholds["calibrated_on"] == "OOF train probabilities only"

print(f"Modelo congelado: {feature_payload['model_name']}")
print(f"Features esperadas: {len(FINAL_FEATURES)}")
print(f"Threshold A1: {THRESHOLD_A1:.2f}")
print(f"Politica: {thresholds['policy']}")

## 3. Reconstruccion de la matriz final de test

Se reconstruye la matriz de evaluacion usando solo artefactos ya generados. Las columnas se seleccionan en el orden exacto guardado en `feature_list.json`. Si falta alguna columna, el notebook se detiene porque la evaluacion ya no seria equivalente al modelo congelado.

In [ ]:
X_test_base = pd.read_parquet(INPUTS["X_test"]).reset_index(drop=True)
y_test = pd.read_parquet(INPUTS["y_test"]).squeeze().astype(int).reset_index(drop=True)

assert len(X_test_base) == len(y_test)
assert y_test.isin(CLASSES).all()
assert X_test_base.index.equals(y_test.index)
assert not X_test_base.columns.duplicated().any()

blocks: dict[str, pd.DataFrame] = {"base": X_test_base}
for name, path in [
    ("exp5b", INPUTS["X_test_exp5b"]),
    ("llm", INPUTS["llm_test"]),
    ("bert", INPUTS["bert_test"]),
]:
    if path.exists():
        block = pd.read_parquet(path).reset_index(drop=True)
        assert len(block) == len(y_test), name
        assert not block.columns.duplicated().any(), name
        blocks[name] = block

if any(col.startswith("bert_svd_") for col in FINAL_FEATURES):
    bert_svd_path = (
        INPUTS["bert_svd"] if INPUTS["bert_svd"].exists() else INPUTS["bert_svd_legacy"]
    )
    if not bert_svd_path.exists():
        raise FileNotFoundError(
            "El modelo final usa BERT/SVD pero no existe bert_svd.joblib"
        )
else:
    bert_svd_path = None

selected_parts = []
selected_columns = []
for feature in FINAL_FEATURES:
    found = [
        (name, block) for name, block in blocks.items() if feature in block.columns
    ]
    if not found:
        raise ValueError(f"Falta la feature final en test: {feature}")
    block_name, block = found[-1]
    selected_parts.append(block[[feature]].reset_index(drop=True))
    selected_columns.append(feature)

X_eval = pd.concat(selected_parts, axis=1)
X_eval.columns = selected_columns

assert len(X_eval) == len(y_test)
assert list(X_eval.columns) == FINAL_FEATURES
assert not X_eval.columns.duplicated().any()
assert not X_eval.isna().any().any()
assert model.n_features_in_ == X_eval.shape[1]

print(f"X_eval: {X_eval.shape}")
print(f"y_test: {y_test.shape}")
bert_svd_label = (
    str(bert_svd_path.relative_to(PROJECT_ROOT)) if bert_svd_path else "no aplica"
)
print(f"BERT/SVD: {bert_svd_label}")

## 4. Inferencia final

Se calculan probabilidades con el modelo congelado. Se muestran dos decisiones: `argmax`, como referencia directa del modelo, y la politica final, que prioriza la seguridad A1 antes de aplicar la ponderacion calibrada en OOF.

In [ ]:
def aplicar_politica_clinica(
    proba: np.ndarray,
    class_weights: np.ndarray,
    threshold_a1: float,
) -> np.ndarray:
    assert threshold_a1 >= 0.20
    weighted_pred = np.argmax(proba * class_weights.reshape(1, -1), axis=1) + 1
    final_pred = weighted_pred.astype(int)
    final_pred[proba[:, 0] >= threshold_a1] = 1
    return final_pred


proba_test = model.predict_proba(X_eval)
y_pred_argmax = np.argmax(proba_test, axis=1) + 1
y_pred_final = aplicar_politica_clinica(proba_test, CLASS_WEIGHTS, THRESHOLD_A1)

assert proba_test.shape == (len(y_test), 5)
assert np.allclose(proba_test.sum(axis=1), 1.0, atol=1e-5)
assert set(y_pred_argmax).issubset(set(CLASSES))
assert set(y_pred_final).issubset(set(CLASSES))

changed_by_policy = int((y_pred_argmax != y_pred_final).sum())
print(f"Casos modificados por la politica final: {changed_by_policy:,}")

## 5. Metricas finales

Las metricas se calculan una sola vez sobre test temporal. Macro F1 resume el rendimiento equilibrando clases, mientras que Weighted F1 refleja mejor el peso de las clases frecuentes. Balanced Accuracy y las metricas por clase ayudan a leer el resultado bajo desbalance.

In [ ]:
def resumen_metricas(y_true: pd.Series, y_pred: np.ndarray) -> dict[str, Any]:
    precision, recall, f1, support = precision_recall_fscore_support(
        y_true, y_pred, labels=CLASSES, zero_division=0
    )
    metrics = {
        "macro_f1": float(f1_score(y_true, y_pred, average="macro", zero_division=0)),
        "weighted_f1": float(
            f1_score(y_true, y_pred, average="weighted", zero_division=0)
        ),
        "balanced_accuracy": float(balanced_accuracy_score(y_true, y_pred)),
    }
    for i, cls in enumerate(CLASSES):
        metrics[f"acuity_{cls}_precision"] = float(precision[i])
        metrics[f"acuity_{cls}_recall"] = float(recall[i])
        metrics[f"acuity_{cls}_f1"] = float(f1[i])
        metrics[f"acuity_{cls}_support"] = int(support[i])
    return metrics


metrics_argmax = resumen_metricas(y_test, y_pred_argmax)
metrics_final = resumen_metricas(y_test, y_pred_final)
report_df = pd.DataFrame(
    classification_report(
        y_test,
        y_pred_final,
        labels=CLASSES,
        target_names=CLASS_NAMES,
        zero_division=0,
        output_dict=True,
    )
).T
report_df.to_csv(OUTPUTS["classification_report"], index=True)

final_metrics = {
    "model_name": feature_payload["model_name"],
    "n_test": int(len(y_test)),
    "decision_argmax": metrics_argmax,
    "decision_final_policy": metrics_final,
    "threshold_a1": THRESHOLD_A1,
    "class_weights": [float(x) for x in CLASS_WEIGHTS],
}
OUTPUTS["final_metrics"].write_text(
    json.dumps(final_metrics, indent=2, ensure_ascii=False), encoding="utf-8"
)

pd.DataFrame([metrics_argmax, metrics_final], index=["argmax", "politica_final"])

## 6. Matrices de confusion

La matriz absoluta permite ver el volumen real de errores. La normalizada por fila ayuda a interpretar que parte de cada Acuity acaba en cada prediccion. Los errores por encima de la diagonal implican infratriaje; por debajo, sobretriaje.

In [ ]:
def guardar_matriz_confusion(
    normalize: str | None, path: Path, title: str
) -> pd.DataFrame:
    cm = confusion_matrix(y_test, y_pred_final, labels=CLASSES, normalize=normalize)
    fig, ax = plt.subplots(figsize=(7, 6))
    im = ax.imshow(cm, cmap="Blues")
    ax.set_title(title)
    ax.set_xlabel("Prediccion")
    ax.set_ylabel("Acuity real")
    ax.set_xticks(range(len(CLASSES)), labels=CLASSES)
    ax.set_yticks(range(len(CLASSES)), labels=CLASSES)
    fmt = ".2f" if normalize else "d"
    values = cm if normalize else cm.astype(int)
    for i in range(values.shape[0]):
        for j in range(values.shape[1]):
            ax.text(j, i, format(values[i, j], fmt), ha="center", va="center")
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    fig.tight_layout()
    fig.savefig(path, dpi=200, bbox_inches="tight")
    plt.close(fig)
    return pd.DataFrame(cm, index=CLASSES, columns=CLASSES)


cm_abs = guardar_matriz_confusion(
    None, OUTPUTS["confusion_matrix"], "Matriz de confusion final"
)
cm_norm = guardar_matriz_confusion(
    "true", OUTPUTS["confusion_matrix_normalized"], "Matriz de confusion normalizada"
)
cm_abs

## 7. Analisis clinico de seguridad

El punto central de seguridad es Acuity 1. Se cuantifican rescates por la politica A1, falsas alarmas y casos criticos que aun quedan predichos como niveles inferiores. Este analisis es descriptivo; no cambia thresholds.

In [ ]:
a1_mask = y_test.to_numpy() == 1
policy_changed = y_pred_argmax != y_pred_final
rescued_a1 = a1_mask & (y_pred_argmax != 1) & (y_pred_final == 1)
false_alarm_a1 = (~a1_mask) & (y_pred_final == 1)
missed_a1_final = a1_mask & (y_pred_final != 1)
undertriage = y_pred_final > y_test.to_numpy()
overtriage = y_pred_final < y_test.to_numpy()
critical_undertriage = a1_mask & undertriage

safety_rows = [
    {"indicador": "a1_reales", "n": int(a1_mask.sum())},
    {"indicador": "rescates_a1_por_politica", "n": int(rescued_a1.sum())},
    {"indicador": "falsas_alarmas_a1", "n": int(false_alarm_a1.sum())},
    {"indicador": "a1_infratriados_final", "n": int(missed_a1_final.sum())},
    {"indicador": "infratriaje_total", "n": int(undertriage.sum())},
    {"indicador": "sobretriaje_total", "n": int(overtriage.sum())},
    {"indicador": "infratriaje_critico_a1", "n": int(critical_undertriage.sum())},
    {"indicador": "casos_modificados_por_politica", "n": int(policy_changed.sum())},
]
safety_df = pd.DataFrame(safety_rows)
safety_df["pct_test"] = safety_df["n"] / len(y_test)
safety_df.to_csv(OUTPUTS["safety_analysis"], index=False)
safety_df

## 8. AUPRC frente al desbalance

Acuity 1 y Acuity 2 son clases clinicamente relevantes y no dominan el dataset. AUPRC resume la calidad de ranking para cada clase positiva y es mas informativa que la accuracy cuando hay desbalance.

In [ ]:
y_bin = label_binarize(y_test, classes=CLASSES)
auprc = {
    "acuity_1_auprc": float(average_precision_score(y_bin[:, 0], proba_test[:, 0])),
    "acuity_2_auprc": float(average_precision_score(y_bin[:, 1], proba_test[:, 1])),
}
OUTPUTS["auprc_metrics"].write_text(
    json.dumps(auprc, indent=2, ensure_ascii=False), encoding="utf-8"
)
auprc

## 9. Explicabilidad SHAP

SHAP se usa aqui solo como herramienta descriptiva del modelo final. No se usa para seleccionar variables ni para cambiar la configuracion. Para que sea reproducible y manejable, se toma una muestra fija del test.

In [ ]:
SHAP_SAMPLE_SIZE = min(2000, len(X_eval))
rng = np.random.default_rng(RANDOM_STATE)
shap_idx = np.sort(rng.choice(len(X_eval), size=SHAP_SAMPLE_SIZE, replace=False))
X_shap = X_eval.iloc[shap_idx]

try:
    import shap

    explainer = shap.TreeExplainer(model)
    shap_values = explainer.shap_values(X_shap)
    if isinstance(shap_values, list):
        shap_for_summary = np.mean(np.abs(np.stack(shap_values, axis=0)), axis=0)
    elif getattr(shap_values, "ndim", 0) == 3:
        shap_for_summary = np.mean(np.abs(shap_values), axis=2)
    else:
        shap_for_summary = shap_values

    plt.figure(figsize=(9, 7))
    shap.summary_plot(shap_for_summary, X_shap, show=False, max_display=25)
    plt.title("SHAP summary - modelo final")
    plt.tight_layout()
    plt.savefig(OUTPUTS["shap_summary"], dpi=200, bbox_inches="tight")
    plt.close()
    print(f"SHAP guardado en {OUTPUTS['shap_summary'].relative_to(PROJECT_ROOT)}")
except Exception as exc:
    print(f"SHAP no se pudo calcular en esta ejecucion: {exc}")

## 10. Limitaciones

La evaluacion temporal mide el rendimiento en MIMIC-IV-ED, no en uso clinico real. En produccion academica el sistema depende de extraccion LLM conservadora desde texto libre, lo que introduce distribution shift respecto a variables estructuradas. Tambien existen limitaciones por abreviaturas, idioma del texto, diferencias entre ingles y espanol, defaults clinicos cuando falta informacion y ausencia de validacion prospectiva. El prototipo no debe usarse para tomar decisiones clinicas reales.

## 11. Conclusion final

El notebook evalua una configuracion ya seleccionada: LightGBM con la lista exacta de variables congelada y politica clinica A1. Las fortalezas principales son la separacion train/test, el uso de OOF para calibrar antes de test y la proteccion explicita de pacientes criticos. Las limitaciones principales son el posible cambio de distribucion en produccion, la dependencia del texto libre y la naturaleza academica del sistema. Las lineas futuras razonables serian validacion externa, revision clinica de errores A1/A2 y monitorizacion de calidad de extraccion LLM.